# Stage 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/slotA-rag-harness'
for sub in ['artifacts/index', 'artifacts/runs', 'artifacts/answers',
            'artifacts/scores', 'artifacts/report', 'notes', 'reference']:
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)
os.chdir(PROJECT)
print(os.getcwd())
!ls -R artifacts | head -20

In [ ]:
!pip install -r requirements-colab.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 16.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.2 MB/s eta 0:00:00
 

In [ ]:
!nvidia-smi

In [ ]:
!python env_check.py

# Stage 1

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/slotA-rag-harness'
os.environ['SLOTA_ROOT'] = PROJECT     # so src/common.py resolves paths to Drive
os.chdir(PROJECT)

In [ ]:
!python src/01_index.py


Stage 1 — SKIPPED (artifacts already present)
  artifacts/index/faiss.index  (16983 KiB)
  artifacts/index/bm25.pkl  (9027 KiB)
  artifacts/index/docmap.json  (15658 KiB)
  artifacts/index/index_meta.json  (4 KiB)

  built at        : 2026-08-06T04:35:30+00:00
  embedder        : BAAI/bge-base-en-v1.5
  chunk_policy    : chunked   dedup_needed=True
  faiss ntotal    : 5661  dim=768

  Pass --force to rebuild. Nothing was changed.


In [ ]:
!python src/verify_stage1.py


[3] Cold reload from disk (fresh process)
  faiss.read_index    : OK  ntotal=5661  dim=768  type=IndexFlatIP
  bm25 unpickle       : OK  docs=5183  rank_bm25=0.2.2
  docmap.json         : OK  n_units=5661  policy=chunked  dedup_needed=True
  index_meta.json     : OK  built 2026-08-06T04:35:30+00:00
      faiss.index          16.59 MiB
      bm25.pkl              8.82 MiB
      docmap.json          15.29 MiB
      index_meta.json       0.00 MiB

[1] Corpus counts (re-read from the cached BEIR download)
100% 5183/5183 [00:00<00:00, 98444.38it/s]
  corpus docs         : 5183  (published 5183)
  test queries        : 300   (published 300)
  qrels pairs         : 339
  -> matches published stats

[2] FAISS shape assertions
  index.ntotal        : 5661
  index.d             : 768  (expect 768 for bge-base)
  docmap n_units      : 5661
  meta n_units        : 5661
  OK    ntotal == docmap n_units
  OK    ntotal == meta n_units
  OK    ntotal == len(ordinal_to_docid)
  OK    index.d == 768
  

# Stage 2

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/slotA-rag-harness'
os.environ['SLOTA_ROOT'] = PROJECT     # so src/common.py resolves paths to Drive
os.chdir(PROJECT)

In [ ]:
# Execute candidate retrieval & metric evaluation across all 3 arms
!python src/02_retrieve.py


[1] Load Stage 1 artifacts & SciFact test dataset
100% 5183/5183 [00:00<00:00, 99026.45it/s]
  test queries        : 300
  corpus docs         : 5183
  docmap units        : 5661 (dedup_needed=True)
  FAISS ntotal        : 5661
2026-08-06 07:18:24.088518: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

[2] BM25 Candidate Retrieval (Sparse Arm)
  BM25 retrieval total: 6.10s (20.28 ms/query avg)

[3] Dense Candidate Retrieval (BAAI/bge-base-en-v1.5 + Max-Collapse)
  Dense embed time    : 0.62s (2.08 ms/query)
  Dense FAISS search  : 0.33s (1.11 ms/query avg)
  Total Dense latency : 3.19 ms/query avg

[4] RRF Hybrid Candidate Retrieval (BM25 + Dense Fusion, k=60)
  RRF fusion total    : 18.91 ms (0.0630 ms/query avg)

[5] Compute Hand-Written Retr

In [ ]:
# Verify outputs & run independent library parity check in fresh process
!python src/verify_stage2.py


[1] Canary Math Verification
  Canary metric calculations : PASS

[2] Cold Reload Candidate Runs from disk
  Loaded : retrieval_bm25.json (1499.8 KiB)
  Loaded : retrieval_dense.json (1505.0 KiB)
  Loaded : retrieval_rrf.json (1523.3 KiB)
  Loaded : stage2_retrieval_metrics.json (2.0 KiB)
100% 5183/5183 [00:00<00:00, 56813.25it/s]

[3] Validate Candidate Structure & Docid Resolution
  BM25         : queries=300  candidates/query=50 (OK)  unresolved_docs=0
  Dense        : queries=300  candidates/query=50 (OK)  unresolved_docs=0
  RRF Hybrid   : queries=300  candidates/query=50 (OK)  unresolved_docs=0

[4] Metric Library Parity Cross-Validation

  Checking BM25 arm:
  pytrec_eval parity  : OK (Rec@10: 0.6862, MRR: 0.5288, nDCG@10: 0.5597)
  ranx not installed — skipping library parity check

  Checking Dense arm:
  pytrec_eval parity  : OK (Rec@10: 0.8709, MRR: 0.7085, nDCG@10: 0.7407)
  ranx not installed — skipping library parity check

  Checking RRF Hybrid arm:
  pytrec_eval parity

# Stage 3

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/slotA-rag-harness'
os.environ['SLOTA_ROOT'] = PROJECT     # so src/common.py resolves paths to Drive
os.chdir(PROJECT)

In [ ]:
# Execute staged multi-model reranking across all 3 candidate sets
!python src/03_rerank.py


[1] Load Stage 2 Candidate Runs & SciFact Document Corpus
100% 5183/5183 [00:00<00:00, 96105.56it/s]

[2] Stage 3A: Load bge-reranker-v2-m3 & Rerank Candidates
  [SKIPPED - CACHED] rerank_m3_bm25.json (nDCG@10: 0.6693, rerank: 1781.12 ms/query)
  [SKIPPED - CACHED] rerank_m3_dense.json (nDCG@10: 0.7420, rerank: 1828.61 ms/query)
  [SKIPPED - CACHED] rerank_m3_rrf.json (nDCG@10: 0.7347, rerank: 1816.43 ms/query)

[3] Stage 3B: Load bge-reranker-v2-gemma & Rerank Candidates
  Loading BAAI/bge-reranker-v2-gemma onto CUDA...
2026-08-06 15:18:35.734986: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 1.11kB [00:00, 4.90MB/s]
tokenizer.model: 100% 4.24M/4.24M [00:00<00:00, 16.9MB/s]
tokenizer.json: 100% 17.5M/17.5M [00:00<00:00,

In [ ]:
# Verify all 9 grid cells, pytrec_eval parity & Pareto frontier in fresh process
!python src/verify_stage3.py


[1] Cold Reload All 9 Run JSON Artifacts from disk
  Loaded : retrieval_bm25.json          ( 1499.8 KiB)
  Loaded : retrieval_dense.json         ( 1505.0 KiB)
  Loaded : retrieval_rrf.json           ( 1523.3 KiB)
  Loaded : rerank_m3_bm25.json          ( 1539.0 KiB)
  Loaded : rerank_m3_dense.json         ( 1537.3 KiB)
  Loaded : rerank_m3_rrf.json           ( 1536.8 KiB)
  Loaded : rerank_gemma_bm25.json       ( 1516.6 KiB)
  Loaded : rerank_gemma_dense.json      ( 1514.0 KiB)
  Loaded : rerank_gemma_rrf.json        ( 1514.0 KiB)
  Loaded : stage3_rerank_metrics.json   (    4.6 KiB)
100% 5183/5183 [00:00<00:00, 94321.70it/s]

[2] Validate Candidate Structure & Docid Resolution across 9 Runs
  bm25                             : queries=300 candidates/q=50 (OK) unresolved=0
  dense                            : queries=300 candidates/q=50 (OK) unresolved=0
  rrf_hybrid                       : queries=300 candidates/q=50 (OK) unresolved=0
  BM25 -> bge-v2-m3                : queries=300 

# Comparing Dense Models to determine what should be used

## Comparing BAAI/bge-base-en-v1.5 Vs intfloat/e5-large-v2

In [ ]:
!python src/01b_index_e5.py

2026-08-06 22:45:49.965070: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

[1] Load SciFact Corpus for e5-large-v2 Indexing
100% 5183/5183 [00:00<00:00, 94282.43it/s]

[2] Load intfloat/e5-large-v2 & Embed Passages (fp16 on CUDA)
  model               : intfloat/e5-large-v2
  dim                 : 1024
  passage prefix      : 'passage: '
modules.json: 100% 387/387 [00:00<00:00, 2.02MB/s]
README.md: 67.8kB [00:00, 73.8MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 303kB/s]
config.json: 100% 616/616 [00:00<00:00, 4.81MB/s]
model.safetensors: 100% 1.34G/1.34G [00:12<00:00, 106MB/s]
tokenizer_config.json: 100% 314/314 [00:00<00:00, 2.53MB/s]
vocab.txt: 232kB [00:00, 20.6MB/s]
tokenizer.json: 711kB [00:00, 10.1MB/s]
special_tokens_map

In [ ]:
!python src/02b_retrieve_e5.py

2026-08-06 22:48:10.711118: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

[1] Load e5 Index & SciFact Test Queries
100% 5183/5183 [00:00<00:00, 49459.38it/s]
  e5 FAISS ntotal     : 5183 (dim: 1024)
  test queries        : 300

[2] Encode Queries with 'query: ' Prefix & Retrieve Top-50

[3] Evaluate e5-large-v2 Metrics & Compare vs bge-base-en-v1.5

Dense Embedding Model Comparison Baseline (SciFact Test Set):
Model                        | Recall@10  | MRR      | nDCG@10  | P@1    | Latency (ms)
-------------------------------------------------------------------------------------
BAAI/bge-base-en-v1.5        | 0.8709     | 0.7085   | 0.7407   | 0.6200 | 3.19 ms
intfloat/e5-large-v2         | 0.8438     | 0.6936   | 0.7230   | 0.6067 | 3.55   

In [ ]:
!python src/verify_e5.py


[1] Cold Reload e5 Artifacts from disk
  Loaded : faiss_e5.index           (20732.0 KiB)
  Loaded : docmap_e5.json           ( 7851.2 KiB)
  Loaded : index_meta_e5.json       (    0.5 KiB)
  Loaded : retrieval_e5_dense.json  ( 1504.9 KiB)
  Loaded : e5_vs_bge_comparison.json (    0.9 KiB)
100% 5183/5183 [00:00<00:00, 78855.06it/s]

[2] FAISS & Docmap Assertions (e5-large-v2)
  index.ntotal        : 5183 (expect 5183)
  index.d             : 1024 (expect 1024 for e5-large)

[3] Candidate Run Structure & Docid Resolution
  queries in run      : 300
  candidates/query=50 : OK
  unresolved docids   : 0

[4] Dense Embedding Model Comparison Baseline
Model                        | Recall@10  | MRR      | nDCG@10  | P@1   
------------------------------------------------------------------------
BAAI/bge-base-en-v1.5        | 0.8709     | 0.7085   | 0.7407   | 0.6200
intfloat/e5-large-v2         | 0.8438     | 0.6936   | 0.7230   | 0.6067

VERDICT
  All e5 dense embedding checks passed cleanl

## Comparing BAAI/bge-base-en-v1.5 Vs BAAI/bge-small-en-v1.5

In [ ]:
!python src/01c_index_bge_small.py

2026-08-06 22:57:09.630775: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

[1] Load SciFact Corpus for bge-small-en-v1.5 Indexing
100% 5183/5183 [00:00<00:00, 94651.48it/s]

[2] Load BAAI/bge-small-en-v1.5 & Embed Passages (fp16 on CUDA)
  model               : BAAI/bge-small-en-v1.5
  dim                 : 384
  parameters          : ~33M
modules.json: 100% 349/349 [00:00<00:00, 2.18MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 1.27MB/s]
README.md: 94.8kB [00:00, 83.0MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 368kB/s]
config.json: 100% 743/743 [00:00<00:00, 7.30MB/s]
model.safetensors: 100% 133M/133M [00:06<00:00, 21.5MB/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.88MB/s]
vocab.txt: 232kB [00

In [ ]:
!python src/02c_retrieve_bge_small.py

2026-08-06 22:57:47.704945: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

[1] Load bge-small Index & SciFact Test Queries
100% 5183/5183 [00:00<00:00, 53200.95it/s]
  bge-small FAISS ntotal : 5183 (dim: 384)
  test queries           : 300

[2] Embed Queries with BGE Instruction & Retrieve Top-50

[3] Compute bge-small Metrics & Assemble 3-Tier Sweet-Spot Matrix

3-Tier Dense Embedding Model Size Benchmark (Small vs Base vs Large):
Tier                   | Model                    | Dim   | Recall@10  | MRR      | nDCG@10  | P@1   
-----------------------------------------------------------------------------------------------
Small (33M params)     | BAAI/bge-small-en-v1.5   | 384   | 0.8362     | 0.6864   | 0.7127   | 0.6067
Base (109M params)

In [ ]:
!python src/verify_bge_small.py


[1] Cold Reload bge-small Artifacts from disk
  Loaded : faiss_bge_small.index          ( 7774.5 KiB)
  Loaded : docmap_bge_small.json          ( 7851.2 KiB)
  Loaded : index_meta_bge_small.json      (    0.5 KiB)
  Loaded : retrieval_bge_small_dense.json ( 1505.0 KiB)
  Loaded : dense_model_size_sweetspot.json (    0.9 KiB)
100% 5183/5183 [00:00<00:00, 92285.23it/s]

[2] FAISS & Docmap Assertions (bge-small-en-v1.5)
  index.ntotal        : 5183 (expect 5183)
  index.d             : 384 (expect 384 for bge-small)

[3] Candidate Run Structure & Docid Resolution
  queries in run      : 300
  candidates/query=50 : OK
  unresolved docids   : 0

[4] Render 3-Tier Dense Model Size Sweet-Spot Benchmark
Tier                   | Model                    | Dim   | Recall@10  | MRR      | nDCG@10  | P@1   
-----------------------------------------------------------------------------------------------
Small (33M params)     | BAAI/bge-small-en-v1.5   | 384   | 0.8362     | 0.6864   | 0.7127   | 0

# Phase 2 - Stage 4: Adding Answer GenLLM

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/slotA-rag-harness'
os.environ['SLOTA_ROOT'] = PROJECT     # so src/common.py resolves paths to Drive
os.chdir(PROJECT)

In [ ]:
!python src/04_generate.py --force


[1] Load SciFact Dataset & Document Corpus
100% 5183/5183 [00:00<00:00, 55079.42it/s]

[2] Load Qwen/Qwen2.5-1.5B-Instruct Generator onto CUDA
  Loading Qwen/Qwen2.5-1.5B-Instruct...
tokenizer_config.json: 7.30kB [00:00, 36.6MB/s]
vocab.json: 2.78MB [00:00, 69.9MB/s]
merges.txt: 1.67MB [00:00, 132MB/s]
tokenizer.json: 7.03MB [00:00, 179MB/s]
config.json: 100% 660/660 [00:00<00:00, 5.77MB/s]
2026-08-07 23:24:11.958010: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
model.safetensors: 100% 3.09G/3.09G [01:03<00:00, 48.9MB/s]
generation_config.json: 100% 242/242 [00:00<00:00, 2.66MB/s]

[3] Execute Local Answer Generation across Survivor Runs (Per-Query Auto-Save)

  Generating answers for: Dense -> none...
  Generating [Dense -> none]: 100% 300/3

In [ ]:
!python src/verify_stage4.py


[1] Cold Reload All 4 Survivor Generation Artifacts from disk
  Loaded : gen_dense_none.json            (  301.8 KiB)
  Loaded : gen_dense_m3.json              (  297.9 KiB)
  Loaded : gen_rrf_gemma.json             (  302.2 KiB)
  Loaded : gen_dense_gemma.json           (  298.8 KiB)
  Loaded : stage4_generation_summary.json (    0.9 KiB)
100% 5183/5183 [00:00<00:00, 86297.34it/s]

[2] Validate Answer Completeness & Citation Integrity across Survivor Runs
  Dense -> none                : answers=300 empty=0 (OK) cited=1.3% mean_len=91.0 words truncs=4
  Dense -> bge-v2-m3           : answers=300 empty=0 (OK) cited=2.7% mean_len=89.0 words truncs=5
  RRF Hybrid -> bge-v2-gemma   : answers=300 empty=0 (OK) cited=3.3% mean_len=90.8 words truncs=5
  Dense -> bge-v2-gemma        : answers=300 empty=0 (OK) cited=3.3% mean_len=89.5 words truncs=5

[3] Render Stage 4 Generation Latency & Summary Table
Config                       | Output File              | Gen Latency (ms)   | Truncations


# Stage 5

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/slotA-rag-harness'
os.environ['SLOTA_ROOT'] = PROJECT     # so src/common.py resolves paths to Drive
os.chdir(PROJECT)

In [ ]:
!python src/05_judge.py --force


[1] Load SciFact Document Corpus & Stage 3 Rerank Metrics

[2] Load Local LLM Judge onto CUDA
tokenizer_config.json: 7.30kB [00:00, 33.0MB/s]
vocab.json: 2.78MB [00:00, 65.6MB/s]
merges.txt: 1.67MB [00:00, 144MB/s]
tokenizer.json: 7.03MB [00:00, 170MB/s]
config.json: 100% 663/663 [00:00<00:00, 6.03MB/s]
2026-08-08 09:58:47.168471: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  4-bit bitsandbytes load failed (Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
No module named 'triton.ops') — falling back to Qwen2.5-3B-Instruct in fp16...
  Loading judge model: Qwen/Qwen2.5-3B-Instruct (fp16 on CUDA)...
tokenizer_config.json: 7.30kB [00:00, 3.94MB/s]
vocab.json: 2.78MB [00:00, 1

In [ ]:
!python src/verify_stage5.py


[1] Cold Reload All 4 Survivor Judgment Artifacts from disk
  Loaded : judge_dense_none.json          (  332.0 KiB)
  Loaded : judge_dense_m3.json            (  328.7 KiB)
  Loaded : judge_rrf_gemma.json           (  332.3 KiB)
  Loaded : judge_dense_gemma.json         (  329.4 KiB)
  Loaded : stage5_faithfulness_summary.json (    1.2 KiB)
100% 5183/5183 [00:00<00:00, 86907.64it/s]

[2] Validate Judgment Completeness & Score Bounds across Survivor Runs
  Dense -> none                : judgments=300 invalid=0 (OK) mean_faithfulness=0.5388
  Dense -> bge-v2-m3           : judgments=300 invalid=0 (OK) mean_faithfulness=0.5642
  RRF Hybrid -> bge-v2-gemma   : judgments=300 invalid=0 (OK) mean_faithfulness=0.5713
  Dense -> bge-v2-gemma        : judgments=300 invalid=0 (OK) mean_faithfulness=0.5497

[3] Render Final Phase 2 Pareto Survivor Matrix
Config                       | Retrieval nDCG@10  | Total Latency (ms)   | Faithfulness
---------------------------------------------------------